Experiment 6 (E6): a sentence-level hierarchical model that looks for changes in writing style across a document.

The central idea is to represent a document as a sequence of sentence embeddings, then combine:
- sentence content — what each sentence says;
- document-level structure — how sentence representations relate to one another;
- inconsistency features — changes between adjacent sentences and overall variation across the document.
I’ve kept it validation-only: train on the designated 40k training file, evaluate on val_fixed.csv, and save the model and validation outputs. There is no test inference or submission code.

In [1]:
import os, re, json, math, random, gc
from pathlib import Path

import numpy as np
import pandas as pd

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader

from transformers import AutoTokenizer, AutoModel, get_linear_schedule_with_warmup
from sklearn.metrics import (
    f1_score, classification_report, confusion_matrix,
    accuracy_score
)
from tqdm.auto import tqdm

SEED = 42

def seed_everything(seed=SEED):
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    torch.cuda.manual_seed_all(seed)

seed_everything()

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print("Device:", DEVICE)

TRAIN_PATH = "/kaggle/input/datasets/froggyyfrograss/base-train-data/train_transformer_40k.csv"
VAL_PATH   = "/kaggle/input/datasets/froggyyfrograss/base-train-data/val_fixed.csv"

OUT_DIR = Path("/kaggle/working/e6_sentence_hierarchy_artifacts")
OUT_DIR.mkdir(parents=True, exist_ok=True)

MODEL_NAME = "xlm-roberta-base"
LABELS = ["human", "ai", "fusion"]
LABEL2ID = {name: i for i, name in enumerate(LABELS)}
ID2LABEL = {i: name for name, i in LABEL2ID.items()}

MAX_SENTENCES = 12
MAX_SENTENCE_TOKENS = 128

EPOCHS = 3
BATCH_SIZE = 2
GRAD_ACCUM = 4
ENCODER_LR = 1e-5
HEAD_LR = 5e-5
WEIGHT_DECAY = 0.01
WARMUP_RATIO = 0.06
PATIENCE = 1

print("Output:", OUT_DIR)

Device: cuda
Output: /kaggle/working/e6_sentence_hierarchy_artifacts


Cell 2 — Load the fixed datasets
This cell detects common text and label column names. If your CSV uses different names, set TEXT_COL and LABEL_COL manually after loading.

In [2]:
train_df = pd.read_csv(TRAIN_PATH)
val_df = pd.read_csv(VAL_PATH)

print("Train:", train_df.shape)
print("Validation:", val_df.shape)
print("Train columns:", train_df.columns.tolist())

def find_column(df, candidates):
    lower_map = {c.lower(): c for c in df.columns}
    for candidate in candidates:
        if candidate.lower() in lower_map:
            return lower_map[candidate.lower()]
    return None

TEXT_COL = find_column(
    train_df,
    ["text", "document", "content", "full_text", "essay"]
)
LABEL_COL = find_column(
    train_df,
    ["label", "class", "target"]
)

if TEXT_COL is None or LABEL_COL is None:
    raise ValueError(
        f"Could not identify text/label columns. "
        f"Set TEXT_COL and LABEL_COL manually. Columns: {train_df.columns.tolist()}"
    )

if TEXT_COL not in val_df.columns or LABEL_COL not in val_df.columns:
    raise ValueError("Validation CSV must contain the same text and label columns.")

print("Text column:", TEXT_COL)
print("Label column:", LABEL_COL)

def normalize_label(x):
    if isinstance(x, str):
        x = x.strip().lower()
        if x in LABEL2ID:
            return LABEL2ID[x]
    if x in LABEL2ID.values():
        return int(x)
    raise ValueError(f"Unexpected label: {x!r}")

train_df = train_df[[TEXT_COL, LABEL_COL]].dropna().reset_index(drop=True)
val_df = val_df[[TEXT_COL, LABEL_COL]].dropna().reset_index(drop=True)

train_texts = train_df[TEXT_COL].astype(str).tolist()
val_texts = val_df[TEXT_COL].astype(str).tolist()

y_train = np.array([normalize_label(x) for x in train_df[LABEL_COL]])
y_val = np.array([normalize_label(x) for x in val_df[LABEL_COL]])

print("Train labels:", np.bincount(y_train, minlength=3))
print("Validation labels:", np.bincount(y_val, minlength=3))

Train: (40000, 15)
Validation: (40853, 15)
Train columns: ['id', 'text', 'lang', 'domain', 'label', 'n_chars', 'n_words', '_nw', 'ttr', 'rate_em_dash', 'rate_semicolon', 'rate_exclaim', 'rate_question', 'rate_ellipsis', 'rate_comma']
Text column: text
Label column: label
Train labels: [ 9801 17909 12290]
Validation labels: [10010 18291 12552]


Cell 3 — Sentence segmentation and tokenizer
The model samples sentences from across the document rather than taking only the beginning. This is important for long documents: a possible authorship change may occur in the middle or near the end.

In [3]:
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)

def split_sentences(text):
    text = str(text).replace("\r", "\n")
    text = re.sub(r"\n+", "\n", text).strip()

    # Split on sentence-ending punctuation or paragraph boundaries.
    parts = re.split(r'(?<=[.!?])\s+|\n+', text)
    sentences = [p.strip() for p in parts if p.strip()]

    # Avoid creating a sequence from an empty document.
    if not sentences:
        sentences = [text if text else "[EMPTY]"]

    # If there are too many sentences, sample uniformly across the document.
    if len(sentences) > MAX_SENTENCES:
        indices = np.linspace(
            0, len(sentences) - 1, MAX_SENTENCES
        ).round().astype(int)
        sentences = [sentences[i] for i in indices]

    return sentences

# Quick sanity check
for example in train_texts[:3]:
    print(split_sentences(example)[:4])
    print("---")

config.json:   0%|          | 0.00/615 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

sentencepiece.bpe.model:   0%|          | 0.00/5.07M [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/9.10M [00:00<?, ?B/s]

['If he will make a statement on his departmental responsibilities.', 'At present, as the hon.', 'I congratulate IPS on its expansion.', 'Yes, of course I knew the IMF forecasts.']
---
['Who Cares?', 'We mourn the victims, we offer condolences, but how long will it take before the global community truly grapples with the systemic issues that perpetuate these horrors?', "Let’s be clear: this isn't a new problem.", 'Or the Tazreen factory fire in 2012, where over 110 workers perished?']
---
['(CNN) Novak Djokovic extended his winning run to 37 matches this year as he beat world No.', 'Second seed Djokovic triumphed 6-4 6-4 in two hours 12 minutes, despite having seemed to be on his last legs when he overcame Andy Murray in a three-hour epic the previous evening.', '"I was able to perform maybe my best play on clay and I beat the number one in the world on clay.', "He was given more time to rest when the men's and women's finals were delayed due to rain in the Eternal City, and Djokovic a

Cell 4 — Dataset and batch collation
Tokenization happens per batch, avoiding the large RAM cost of storing every sentence's token IDs for the entire dataset.

In [4]:
class SentenceDataset(Dataset):
    def __init__(self, texts, labels):
        self.texts = texts
        self.labels = labels

    def __len__(self):
        return len(self.texts)

    def __getitem__(self, idx):
        sentences = split_sentences(self.texts[idx])
        return {
            "sentences": sentences,
            "label": int(self.labels[idx])
        }

def collate_sentences(batch):
    batch_size = len(batch)
    sentence_lists = [item["sentences"] for item in batch]
    labels = torch.tensor(
        [item["label"] for item in batch],
        dtype=torch.long
    )

    sentence_counts = [len(x) for x in sentence_lists]
    max_s = max(sentence_counts)

    flat_sentences = []
    for sentences in sentence_lists:
        flat_sentences.extend(sentences)

    enc = tokenizer(
        flat_sentences,
        padding="max_length",
        truncation=True,
        max_length=MAX_SENTENCE_TOKENS,
        return_tensors="pt"
    )

    input_ids = torch.zeros(
        batch_size, MAX_SENTENCES, MAX_SENTENCE_TOKENS,
        dtype=torch.long
    )
    attention_mask = torch.zeros_like(input_ids)
    sentence_mask = torch.zeros(
        batch_size, MAX_SENTENCES, dtype=torch.bool
    )

    cursor = 0
    for i, sentences in enumerate(sentence_lists):
        count = len(sentences)
        input_ids[i, :count] = enc["input_ids"][cursor:cursor + count]
        attention_mask[i, :count] = enc["attention_mask"][cursor:cursor + count]
        sentence_mask[i, :count] = True
        cursor += count

    return {
        "input_ids": input_ids,
        "attention_mask": attention_mask,
        "sentence_mask": sentence_mask,
        "labels": labels
    }

train_ds = SentenceDataset(train_texts, y_train)
val_ds = SentenceDataset(val_texts, y_val)

train_loader = DataLoader(
    train_ds,
    batch_size=BATCH_SIZE,
    shuffle=True,
    num_workers=0,
    collate_fn=collate_sentences,
    pin_memory=(DEVICE.type == "cuda")
)

val_loader = DataLoader(
    val_ds,
    batch_size=BATCH_SIZE,
    shuffle=False,
    num_workers=0,
    collate_fn=collate_sentences,
    pin_memory=(DEVICE.type == "cuda")
)

print("Train batches:", len(train_loader))
print("Validation batches:", len(val_loader))

Train batches: 20000
Validation batches: 20427


Cell 5 — E6 hierarchical model
Each sentence is encoded by XLM-R. A bidirectional GRU then models the sequence of sentence representations. Attention pooling creates a document representation.

The inconsistency features are:
- mean cosine distance between adjacent sentence embeddings;
- standard deviation of adjacent cosine similarities;
- average distance of sentence embeddings from the document's mean embedding.
These are learned alongside the text representation; they are not treated as standalone authorship rules.

In [5]:
class E6SentenceHierarchy(nn.Module):
    def __init__(self, model_name=MODEL_NAME, hidden_size=256, dropout=0.2):
        super().__init__()

        self.encoder = AutoModel.from_pretrained(model_name)
        encoder_dim = self.encoder.config.hidden_size

        self.sentence_projection = nn.Sequential(
            nn.Linear(encoder_dim, hidden_size),
            nn.Tanh(),
            nn.Dropout(dropout)
        )

        self.sequence_model = nn.GRU(
            input_size=hidden_size,
            hidden_size=hidden_size // 2,
            num_layers=1,
            batch_first=True,
            bidirectional=True
        )

        self.attention = nn.Linear(hidden_size, 1)

        # Three scalar inconsistency statistics are concatenated
        # with the attention-pooled document representation.
        self.classifier = nn.Sequential(
            nn.Linear(hidden_size + 3, hidden_size),
            nn.GELU(),
            nn.Dropout(dropout),
            nn.Linear(hidden_size, 3)
        )

        self.dropout = nn.Dropout(dropout)

    def forward(self, input_ids, attention_mask, sentence_mask):
        bsz, n_sent, seq_len = input_ids.shape

        flat_ids = input_ids.view(bsz * n_sent, seq_len)
        flat_mask = attention_mask.view(bsz * n_sent, seq_len)

        enc_out = self.encoder(
            input_ids=flat_ids,
            attention_mask=flat_mask
        )

        # XLM-R's first token representation for each sentence.
        sentence_vecs = enc_out.last_hidden_state[:, 0, :]
        sentence_vecs = self.sentence_projection(sentence_vecs)
        sentence_vecs = sentence_vecs.view(bsz, n_sent, -1)

        sequence_out, _ = self.sequence_model(sentence_vecs)

        scores = self.attention(sequence_out).squeeze(-1)
        scores = scores.masked_fill(~sentence_mask, -1e4)
        weights = torch.softmax(scores, dim=1)

        doc_vec = torch.sum(
            sequence_out * weights.unsqueeze(-1), dim=1
        )

        # Inconsistency features computed from sentence embeddings.
        normed = F.normalize(sentence_vecs, p=2, dim=-1)
        adjacent_sim = (normed[:, 1:] * normed[:, :-1]).sum(dim=-1)

        adjacent_mask = sentence_mask[:, 1:] & sentence_mask[:, :-1]
        adjacent_dist = 1.0 - adjacent_sim
        adjacent_dist = adjacent_dist * adjacent_mask.float()

        denom = adjacent_mask.sum(dim=1).clamp(min=1).float()
        mean_adj_dist = adjacent_dist.sum(dim=1) / denom

        masked_sim = adjacent_sim.masked_fill(~adjacent_mask, 0.0)
        sim_mean = masked_sim.sum(dim=1) / denom
        sim_var = (
            ((adjacent_sim - sim_mean.unsqueeze(1)) ** 2)
            * adjacent_mask.float()
        ).sum(dim=1) / denom
        sim_std = torch.sqrt(sim_var + 1e-8)

        valid = sentence_mask.float().unsqueeze(-1)
        mean_sentence = (
            (sentence_vecs * valid).sum(dim=1)
            / valid.sum(dim=1).clamp(min=1.0)
        )
        sentence_deviation = torch.norm(
            sentence_vecs - mean_sentence.unsqueeze(1),
            dim=-1
        )
        mean_deviation = (
            (sentence_deviation * sentence_mask.float()).sum(dim=1)
            / sentence_mask.float().sum(dim=1).clamp(min=1.0)
        )

        inconsistency = torch.stack(
            [mean_adj_dist, sim_std, mean_deviation], dim=1
        )

        combined = torch.cat(
            [self.dropout(doc_vec), inconsistency], dim=1
        )
        logits = self.classifier(combined)

        return logits

model = E6SentenceHierarchy().to(DEVICE)
print("Model initialized.")

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.12GB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

[transformers] XLMRobertaModel LOAD REPORT from: xlm-roberta-base
Key                       | Status     |  | 
--------------------------+------------+--+-
lm_head.dense.bias        | UNEXPECTED |  | 
lm_head.layer_norm.bias   | UNEXPECTED |  | 
lm_head.bias              | UNEXPECTED |  | 
lm_head.layer_norm.weight | UNEXPECTED |  | 
lm_head.dense.weight      | UNEXPECTED |  | 

Notes:
- UNEXPECTED:	can be ignored when loading from different task/architecture; not ok if you expect identical arch.


Model initialized.


Cell 6 — Optimizer, class weights and scheduler

In [6]:
class_counts = np.bincount(y_train, minlength=3)
class_weights = len(y_train) / (
    3.0 * np.maximum(class_counts, 1)
)
class_weights = torch.tensor(
    class_weights, dtype=torch.float32, device=DEVICE
)

criterion = nn.CrossEntropyLoss(weight=class_weights)

encoder_params = list(model.encoder.parameters())
encoder_param_ids = {id(p) for p in encoder_params}
head_params = [
    p for p in model.parameters()
    if id(p) not in encoder_param_ids
]

optimizer = torch.optim.AdamW(
    [
        {"params": encoder_params, "lr": ENCODER_LR},
        {"params": head_params, "lr": HEAD_LR}
    ],
    weight_decay=WEIGHT_DECAY
)

updates_per_epoch = math.ceil(len(train_loader) / GRAD_ACCUM)
total_steps = updates_per_epoch * EPOCHS
warmup_steps = int(total_steps * WARMUP_RATIO)

scheduler = get_linear_schedule_with_warmup(
    optimizer,
    num_warmup_steps=warmup_steps,
    num_training_steps=total_steps
)

use_amp = DEVICE.type == "cuda"
scaler = torch.cuda.amp.GradScaler(enabled=use_amp)

print("Class weights:", class_weights.detach().cpu().numpy())
print("Planned optimizer steps:", total_steps)

Class weights: [1.3604054  0.74450463 1.0848929 ]
Planned optimizer steps: 15000


/tmp/ipykernel_23/2264462557.py:37: FutureWarning: `torch.cuda.amp.GradScaler(args...)` is deprecated. Please use `torch.amp.GradScaler('cuda', args...)` instead.
  scaler = torch.cuda.amp.GradScaler(enabled=use_amp)


Cell 7 — Validation function

In [7]:
@torch.no_grad()
def evaluate(model, loader):
    model.eval()

    all_logits = []
    all_labels = []

    for batch in tqdm(loader, desc="Validation", leave=False):
        ids = batch["input_ids"].to(DEVICE)
        masks = batch["attention_mask"].to(DEVICE)
        sent_mask = batch["sentence_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(ids, masks, sent_mask)

        all_logits.append(logits.float().cpu().numpy())
        all_labels.append(labels.cpu().numpy())

    logits = np.concatenate(all_logits)
    labels = np.concatenate(all_labels)
    probs = torch.softmax(torch.tensor(logits), dim=1).numpy()
    preds = probs.argmax(axis=1)

    return {
        "logits": logits,
        "probs": probs,
        "preds": preds,
        "labels": labels,
        "macro_f1": f1_score(labels, preds, average="macro"),
        "accuracy": accuracy_score(labels, preds)
    }

Cell 8 — Training loop
The best checkpoint is selected using validation macro-F1. Gradient accumulation reduces the effective batch size requirement.

In [8]:
best_f1 = -1.0
best_epoch = -1
epochs_without_improvement = 0
history = []

for epoch in range(EPOCHS):
    model.train()
    optimizer.zero_grad(set_to_none=True)

    running_loss = 0.0
    progress = tqdm(
        enumerate(train_loader),
        total=len(train_loader),
        desc=f"Epoch {epoch + 1}/{EPOCHS}"
    )

    for step, batch in progress:
        ids = batch["input_ids"].to(DEVICE)
        masks = batch["attention_mask"].to(DEVICE)
        sent_mask = batch["sentence_mask"].to(DEVICE)
        labels = batch["labels"].to(DEVICE)

        with torch.cuda.amp.autocast(enabled=use_amp):
            logits = model(ids, masks, sent_mask)
            loss = criterion(logits, labels)
            loss_for_backward = loss / GRAD_ACCUM

        scaler.scale(loss_for_backward).backward()
        running_loss += loss.item()

        should_update = (
            (step + 1) % GRAD_ACCUM == 0
            or (step + 1) == len(train_loader)
        )

        if should_update:
            scaler.unscale_(optimizer)
            torch.nn.utils.clip_grad_norm_(
                model.parameters(), max_norm=1.0
            )
            scaler.step(optimizer)
            scaler.update()
            scheduler.step()
            optimizer.zero_grad(set_to_none=True)

        progress.set_postfix(loss=f"{loss.item():.4f}")

    train_loss = running_loss / max(1, len(train_loader))
    metrics = evaluate(model, val_loader)

    record = {
        "epoch": epoch + 1,
        "train_loss": train_loss,
        "val_macro_f1": metrics["macro_f1"],
        "val_accuracy": metrics["accuracy"]
    }
    history.append(record)

    print(record)
    print(
        classification_report(
            metrics["labels"],
            metrics["preds"],
            labels=[0, 1, 2],
            target_names=LABELS,
            digits=4,
            zero_division=0
        )
    )

    if metrics["macro_f1"] > best_f1:
        best_f1 = metrics["macro_f1"]
        best_epoch = epoch + 1
        epochs_without_improvement = 0

        torch.save(
            model.state_dict(),
            OUT_DIR / "best_model_state.pt"
        )

        np.save(OUT_DIR / "val_logits.npy", metrics["logits"])
        np.save(OUT_DIR / "val_probs.npy", metrics["probs"])
        np.save(OUT_DIR / "val_preds.npy", metrics["preds"])
        np.save(OUT_DIR / "val_labels.npy", metrics["labels"])

        with open(OUT_DIR / "best_metrics.json", "w") as f:
            json.dump(
                {
                    "best_epoch": best_epoch,
                    "macro_f1": float(best_f1),
                    "accuracy": float(metrics["accuracy"])
                },
                f,
                indent=2
            )
    else:
        epochs_without_improvement += 1

    if epochs_without_improvement >= PATIENCE:
        print("Early stopping.")
        break

    gc.collect()
    if DEVICE.type == "cuda":
        torch.cuda.empty_cache()

pd.DataFrame(history).to_csv(
    OUT_DIR / "training_history.csv", index=False
)

print("Best epoch:", best_epoch)
print("Best validation macro-F1:", best_f1)

Epoch 1/3:   0%|          | 0/20000 [00:00<?, ?it/s]

/tmp/ipykernel_23/932451188.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Validation:   0%|          | 0/20427 [00:00<?, ?it/s]

/tmp/ipykernel_23/733364516.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


{'epoch': 1, 'train_loss': 0.49258073479871217, 'val_macro_f1': 0.7957736541798109, 'val_accuracy': 0.8341859838934718}
              precision    recall  f1-score   support

       human     0.8277    0.5539    0.6637     10010
          ai     0.9295    0.9934    0.9604     18291
      fusion     0.7096    0.8256    0.7632     12552

    accuracy                         0.8342     40853
   macro avg     0.8223    0.7910    0.7958     40853
weighted avg     0.8370    0.8342    0.8271     40853



Epoch 2/3:   0%|          | 0/20000 [00:00<?, ?it/s]

... (5 outputs left out)

/tmp/ipykernel_23/932451188.py:23: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Validation:   0%|          | 0/20427 [00:00<?, ?it/s]

/tmp/ipykernel_23/733364516.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


{'epoch': 3, 'train_loss': 0.2650835598802948, 'val_macro_f1': 0.8303161514052929, 'val_accuracy': 0.8599858027562235}
              precision    recall  f1-score   support

       human     0.8372    0.6453    0.7288     10010
          ai     0.9456    0.9942    0.9693     18291
      fusion     0.7542    0.8356    0.7928     12552

    accuracy                         0.8600     40853
   macro avg     0.8457    0.8250    0.8303     40853
weighted avg     0.8602    0.8600    0.8562     40853

Early stopping.
Best epoch: 2
Best validation macro-F1: 0.8534394554051724


Cell 9 — Reload best checkpoint and final validation evaluation

In [9]:
best_state = torch.load(
    OUT_DIR / "best_model_state.pt",
    map_location=DEVICE
)
model.load_state_dict(best_state)

final_metrics = evaluate(model, val_loader)

print("Best-checkpoint validation macro-F1:",
      final_metrics["macro_f1"])
print("Accuracy:", final_metrics["accuracy"])

print(
    classification_report(
        final_metrics["labels"],
        final_metrics["preds"],
        labels=[0, 1, 2],
        target_names=LABELS,
        digits=4,
        zero_division=0
    )
)

cm = confusion_matrix(
    final_metrics["labels"],
    final_metrics["preds"],
    labels=[0, 1, 2]
)
print("Confusion matrix (rows=true, columns=predicted):")
print(pd.DataFrame(cm, index=LABELS, columns=LABELS))

np.save(OUT_DIR / "final_val_logits.npy", final_metrics["logits"])
np.save(OUT_DIR / "final_val_probs.npy", final_metrics["probs"])
np.save(OUT_DIR / "final_val_preds.npy", final_metrics["preds"])

Validation:   0%|          | 0/20427 [00:00<?, ?it/s]

/tmp/ipykernel_23/733364516.py:14: FutureWarning: `torch.cuda.amp.autocast(args...)` is deprecated. Please use `torch.amp.autocast('cuda', args...)` instead.
  with torch.cuda.amp.autocast(enabled=use_amp):


Best-checkpoint validation macro-F1: 0.8534394554051724
Accuracy: 0.8781729615940078
              precision    recall  f1-score   support

       human     0.8231    0.7162    0.7659     10010
          ai     0.9707    0.9897    0.9801     18291
      fusion     0.7858    0.8449    0.8143     12552

    accuracy                         0.8782     40853
   macro avg     0.8599    0.8502    0.8534     40853
weighted avg     0.8777    0.8782    0.8767     40853

Confusion matrix (rows=true, columns=predicted):
        human     ai  fusion
human    7169     65    2776
ai         75  18102     114
fusion   1466    481   10605


Cell 10 — Save model configuration and tokenizer

In [10]:
tokenizer.save_pretrained(OUT_DIR / "tokenizer")

config = {
    "experiment": "E6_sentence_hierarchy",
    "model_name": MODEL_NAME,
    "labels": LABELS,
    "max_sentences": MAX_SENTENCES,
    "max_sentence_tokens": MAX_SENTENCE_TOKENS,
    "epochs_requested": EPOCHS,
    "batch_size": BATCH_SIZE,
    "gradient_accumulation": GRAD_ACCUM,
    "encoder_lr": ENCODER_LR,
    "head_lr": HEAD_LR,
    "best_epoch": int(best_epoch),
    "best_validation_macro_f1": float(best_f1),
    "text_column": TEXT_COL,
    "label_column": LABEL_COL,
    "train_path": TRAIN_PATH,
    "validation_path": VAL_PATH
}

with open(OUT_DIR / "config.json", "w") as f:
    json.dump(config, f, indent=2)

print("Saved artifacts:")
for p in sorted(OUT_DIR.rglob("*")):
    if p.is_file():
        print(p, f"{p.stat().st_size / (1024**2):.1f} MB")

Saved artifacts:
/kaggle/working/e6_sentence_hierarchy_artifacts/best_metrics.json 0.0 MB
/kaggle/working/e6_sentence_hierarchy_artifacts/best_model_state.pt 1062.9 MB
/kaggle/working/e6_sentence_hierarchy_artifacts/config.json 0.0 MB
/kaggle/working/e6_sentence_hierarchy_artifacts/final_val_logits.npy 0.5 MB
/kaggle/working/e6_sentence_hierarchy_artifacts/final_val_preds.npy 0.3 MB
/kaggle/working/e6_sentence_hierarchy_artifacts/final_val_probs.npy 0.5 MB
/kaggle/working/e6_sentence_hierarchy_artifacts/tokenizer/tokenizer.json 16.3 MB
/kaggle/working/e6_sentence_hierarchy_artifacts/tokenizer/tokenizer_config.json 0.0 MB
/kaggle/working/e6_sentence_hierarchy_artifacts/training_history.csv 0.0 MB
/kaggle/working/e6_sentence_hierarchy_artifacts/val_labels.npy 0.3 MB
/kaggle/working/e6_sentence_hierarchy_artifacts/val_logits.npy 0.5 MB
/kaggle/working/e6_sentence_hierarchy_artifacts/val_preds.npy 0.3 MB
/kaggle/working/e6_sentence_hierarchy_artifacts/val_probs.npy 0.5 MB


Cell 11 — Compact results summary

In [11]:
summary = {
    "experiment": "E6_sentence_hierarchy",
    "best_epoch": int(best_epoch),
    "macro_f1": float(final_metrics["macro_f1"]),
    "accuracy": float(final_metrics["accuracy"]),
    "per_class": classification_report(
        final_metrics["labels"],
        final_metrics["preds"],
        labels=[0, 1, 2],
        target_names=LABELS,
        output_dict=True,
        zero_division=0
    )
}

with open(OUT_DIR / "summary.json", "w") as f:
    json.dump(summary, f, indent=2)

print(json.dumps({
    "experiment": summary["experiment"],
    "best_epoch": summary["best_epoch"],
    "macro_f1": summary["macro_f1"],
    "accuracy": summary["accuracy"]
}, indent=2))

{
  "experiment": "E6_sentence_hierarchy",
  "best_epoch": 2,
  "macro_f1": 0.8534394554051724,
  "accuracy": 0.8781729615940078
}
